In [1]:
from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image
import requests

device = "mps" if torch.cuda.is_available() else "cpu"
print(device)
model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

# Load image
image_path = "./20240425_112716_nist-sand-30-100_27keV_z8mm_n2625_00000.png"
image = Image.open(image_path).convert("RGB")

# Segment using text prompt
inputs = processor(images=image, text="sand", return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Post-process results
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"Found {len(results['masks'])} objects")
# Results contain:
# - masks: Binary masks resized to original image size
# - boxes: Bounding boxes in absolute pixel coordinates (xyxy format)
# - scores: Confidence scores


/Users/xiaoyachong/anaconda3/envs/sam3_test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cpu


Loading weights: 100%|██████████| 1468/1468 [00:01<00:00, 1250.35it/s, Materializing param=vision_encoder.neck.fpn_layers.3.proj2.weight]                       


The OrderedVocab you are attempting to save contains holes for indices [1], your vocabulary could be corrupted !
The OrderedVocab you are attempting to save contains holes for indices [1], your vocabulary could be corrupted !


FileNotFoundError: [Errno 2] No such file or directory: './20240425_112716_nist-sand-30-100_27keV_z8mm_n2625_00000.png'

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Draw box on image for visualization
img_with_box = image.copy()
draw = ImageDraw.Draw(img_with_box)
box = [280, 1050, 880, 1800]
draw.rectangle(box, outline="red", width=3)

# Display inline in notebook
plt.figure(figsize=(10, 8))
plt.imshow(img_with_box)
plt.axis('off')  # Hide axes
plt.title('Image with Bounding Box')
plt.show()

In [ ]:
import numpy as np
import matplotlib
# Box in xyxy format: [x1, y1, x2, y2] in pixel coordinates
# Example: laptop region
box_xyxy = [280, 1050, 880, 1800]
input_boxes = [[box_xyxy]]  # [batch, num_boxes, 4]
input_boxes_labels = [[1]]  # 1 = positive box

inputs = processor(
    images=image,
    input_boxes=input_boxes,
    input_boxes_labels=input_boxes_labels,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Post-process results
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

def overlay_masks(image, masks):
    image = image.convert("RGBA")
    masks = 255 * masks.cpu().numpy().astype(np.uint8)
    
    n_masks = masks.shape[0]
    cmap = matplotlib.colormaps.get_cmap("rainbow").resampled(n_masks)
    colors = [
        tuple(int(c * 255) for c in cmap(i)[:3])
        for i in range(n_masks)
    ]

    for mask, color in zip(masks, colors):
        mask = Image.fromarray(mask)
        overlay = Image.new("RGBA", image.size, color + (0,))
        alpha = mask.point(lambda v: int(v * 0.5))
        overlay.putalpha(alpha)
        image = Image.alpha_composite(image, overlay)
    return image

overlay_masks(image, results["masks"])



In [ ]:
from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

# Load image
image = Image.open("./20240425_112716_nist-sand-30-100_27keV_z8mm_n2625_00000.png").convert("RGB")

# Define boxes
background_box = [0, 0, 300, 300]
air_box = [880, 1550, 1100, 1800]
sand_box = [280, 1050, 880, 1800]

# ===== Segment SAND (no negative boxes) =====
print("Segmenting sand...")
inputs = processor(
    images=image,
    input_boxes=[[sand_box]],  # Only positive box
    input_boxes_labels=[[1]],   # Only positive label
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

sand_results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

# ===== Segment AIR (no negative boxes) =====
print("Segmenting air...")
inputs = processor(
    images=image,
    input_boxes=[[air_box]],   # Only positive box
    input_boxes_labels=[[1]],   # Only positive label
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

air_results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

# ===== Segment BACKGROUND (no negative boxes) =====
print("Segmenting background...")
inputs = processor(
    images=image,
    input_boxes=[[background_box]],  # Only positive box
    input_boxes_labels=[[1]],         # Only positive label
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

background_results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"\n=== Results ===")
print(f"Sand: {len(sand_results['masks'])}")
print(f"Air: {len(air_results['masks'])}")
print(f"Background: {len(background_results['masks'])}")

# Define overlay_masks function
def overlay_masks(image, masks, colors=None):
    """Overlay masks on image with different colors"""
    image = image.convert("RGBA")
    masks = 255 * masks.cpu().numpy().astype(np.uint8)
    
    n_masks = masks.shape[0]
    
    if colors is None:
        cmap = matplotlib.colormaps.get_cmap("rainbow").resampled(n_masks)
        colors = [
            tuple(int(c * 255) for c in cmap(i)[:3])
            for i in range(n_masks)
        ]
    
    for mask, color in zip(masks, colors):
        mask_img = Image.fromarray(mask)
        overlay = Image.new("RGBA", image.size, color + (0,))
        alpha = mask_img.point(lambda v: int(v * 0.5))
        overlay.putalpha(alpha)
        image = Image.alpha_composite(image, overlay)
    
    return image

# Combine all masks
all_masks = []
all_colors = []

if len(sand_results['masks']) > 0:
    all_masks.extend(sand_results['masks'])
    all_colors.extend([(255, 0, 0)] * len(sand_results['masks']))

if len(air_results['masks']) > 0:
    all_masks.extend(air_results['masks'])
    all_colors.extend([(0, 150, 255)] * len(air_results['masks']))

if len(background_results['masks']) > 0:
    all_masks.extend(background_results['masks'])
    all_colors.extend([(0, 255, 0)] * len(background_results['masks']))

# Create visualization
if all_masks:
    combined_masks = torch.stack(all_masks)
    combined_overlay = overlay_masks(image, combined_masks, all_colors)
    
    # CHANGED: Show input boxes on the left image
    img_with_boxes = image.copy()
    draw = ImageDraw.Draw(img_with_boxes)
    draw.rectangle(sand_box, outline="red", width=3)
    draw.rectangle(air_box, outline="blue", width=3)
    draw.rectangle(background_box, outline="green", width=3)
    
    # Display
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    axes[0].imshow(img_with_boxes)  # CHANGED: Show boxes instead of plain image
    axes[0].set_title('Input Boxes: Red=Sand, Blue=Air, Green=BG')  # CHANGED: Updated title
    axes[0].axis('off')
    
    axes[1].imshow(combined_overlay)
    axes[1].set_title(f'Segmentation: Red=Sand({len(sand_results["masks"])}), Blue=Air({len(air_results["masks"])}), Green=BG({len(background_results["masks"])})')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No masks found!")

In [ ]:
from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

# Load image
image = Image.open("./20240425_112716_nist-sand-30-100_27keV_z8mm_n2625_00000.png").convert("RGB")

# ===== Define MULTIPLE boxes for each type =====

# Multiple SAND particle examples
sand_boxes = [
    [280, 1050, 880, 1800],   # Example 1: large sand particle
]

# Multiple AIR region examples
air_boxes = [
    [880, 1550, 1100, 1800],   # Example 1: air region
    [950, 70, 1200, 200],        # Example 2: another air region
]

# Multiple BACKGROUND examples
background_boxes = [
    [0, 0, 300, 300],           # Example 1: background corner
    [2000, 0, 2510, 230],        # Example 2: another background region
]

# ===== Segment SAND with multiple boxes =====
print("Segmenting sand with multiple examples...")
inputs = processor(
    images=image,
    input_boxes=[sand_boxes],  # List of multiple boxes
    input_boxes_labels=[[1] * len(sand_boxes)],  # All positive
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

sand_results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

# ===== Segment AIR with multiple boxes =====
print("Segmenting air with multiple examples...")
inputs = processor(
    images=image,
    input_boxes=[air_boxes],  # List of multiple boxes
    input_boxes_labels=[[1] * len(air_boxes)],  # All positive
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

air_results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

# ===== Segment BACKGROUND with multiple boxes =====
print("Segmenting background with multiple examples...")
inputs = processor(
    images=image,
    input_boxes=[background_boxes],  # List of multiple boxes
    input_boxes_labels=[[1] * len(background_boxes)],  # All positive
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

background_results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"\n=== Results ===")
print(f"Sand: {len(sand_results['masks'])} (from {len(sand_boxes)} examples)")
print(f"Air: {len(air_results['masks'])} (from {len(air_boxes)} examples)")
print(f"Background: {len(background_results['masks'])} (from {len(background_boxes)} examples)")

# Visualization
def overlay_masks(image, masks, colors=None):
    """Overlay masks on image with different colors"""
    image = image.convert("RGBA")
    masks = 255 * masks.cpu().numpy().astype(np.uint8)
    
    n_masks = masks.shape[0]
    
    if colors is None:
        cmap = matplotlib.colormaps.get_cmap("rainbow").resampled(n_masks)
        colors = [
            tuple(int(c * 255) for c in cmap(i)[:3])
            for i in range(n_masks)
        ]
    
    for mask, color in zip(masks, colors):
        mask_img = Image.fromarray(mask)
        overlay = Image.new("RGBA", image.size, color + (0,))
        alpha = mask_img.point(lambda v: int(v * 0.5))
        overlay.putalpha(alpha)
        image = Image.alpha_composite(image, overlay)
    
    return image

# Combine all masks
all_masks = []
all_colors = []

if len(sand_results['masks']) > 0:
    all_masks.extend(sand_results['masks'])
    all_colors.extend([(255, 0, 0)] * len(sand_results['masks']))

if len(air_results['masks']) > 0:
    all_masks.extend(air_results['masks'])
    all_colors.extend([(0, 150, 255)] * len(air_results['masks']))

if len(background_results['masks']) > 0:
    all_masks.extend(background_results['masks'])
    all_colors.extend([(0, 255, 0)] * len(background_results['masks']))

# Create visualization
if all_masks:
    combined_masks = torch.stack(all_masks)
    combined_overlay = overlay_masks(image, combined_masks, all_colors)
    
    # Display with input boxes
    from PIL import ImageDraw
    img_with_boxes = image.copy()
    draw = ImageDraw.Draw(img_with_boxes)
    
    for box in sand_boxes:
        draw.rectangle(box, outline="red", width=2)
    for box in air_boxes:
        draw.rectangle(box, outline="blue", width=2)
    for box in background_boxes:
        draw.rectangle(box, outline="green", width=2)
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    axes[0].imshow(img_with_boxes)
    axes[0].set_title(f'Input Boxes: Red=Sand({len(sand_boxes)}), Blue=Air({len(air_boxes)}), Green=BG({len(background_boxes)})')
    axes[0].axis('off')
    
    axes[1].imshow(combined_overlay)
    axes[1].set_title(f'Results: Red=Sand({len(sand_results["masks"])}), Blue=Air({len(air_results["masks"])}), Green=BG({len(background_results["masks"])})')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
from transformers import Sam3TrackerProcessor, Sam3TrackerModel
import torch
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import numpy as np

# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = Sam3TrackerModel.from_pretrained("facebook/sam3").to(device)
processor = Sam3TrackerProcessor.from_pretrained("facebook/sam3")

# Load image
image = Image.open("./20240425_112716_nist-sand-30-100_27keV_z8mm_n2625_00000.png").convert("RGB")

# Define multiple POINTS for each type [x, y]
sand_points = [
    [580, 1425],  # Sand point 1
]

air_points = [
    [990, 1675],   # Air point 1 (from box [880, 1550, 1100, 1800])
    [1075, 135],   # Air point 2 (from box [950, 70, 1200, 200])
]

background_points = [
    [150, 150],    # Background point 1 (from box [0, 0, 300, 300])
    [2255, 115],   # Background point 2 (from box [2000, 0, 2510, 230])
]

# ===== Segment SAND (positive) with AIR and BACKGROUND (negative) =====
print("Segmenting sand with negative points...")
all_points = sand_points + air_points + background_points
all_labels = [1] * len(sand_points) + [0] * len(air_points) + [0] * len(background_points)

inputs = processor(
    images=image,
    input_points=[[all_points]],   # All points in one list
    input_labels=[[all_labels]],   # Corresponding labels
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs, multimask_output=False)

sand_masks = processor.post_process_masks(
    outputs.pred_masks.cpu(), 
    inputs["original_sizes"]
)[0]

# ===== Segment AIR (positive) with SAND and BACKGROUND (negative) =====
print("Segmenting air with negative points...")
all_points = air_points + sand_points + background_points
all_labels = [1] * len(air_points) + [0] * len(sand_points) + [0] * len(background_points)

inputs = processor(
    images=image,
    input_points=[[all_points]],
    input_labels=[[all_labels]],
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs, multimask_output=False)

air_masks = processor.post_process_masks(
    outputs.pred_masks.cpu(), 
    inputs["original_sizes"]
)[0]

# ===== Segment BACKGROUND (positive) with SAND and AIR (negative) =====
print("Segmenting background with negative points...")
all_points = background_points + sand_points + air_points
all_labels = [1] * len(background_points) + [0] * len(sand_points) + [0] * len(air_points)

inputs = processor(
    images=image,
    input_points=[[all_points]],
    input_labels=[[all_labels]],
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs, multimask_output=False)

background_masks = processor.post_process_masks(
    outputs.pred_masks.cpu(), 
    inputs["original_sizes"]
)[0]

print(f"\n=== Results ===")
print(f"Sand masks: {sand_masks.shape}")
print(f"Air masks: {air_masks.shape}")
print(f"Background masks: {background_masks.shape}")

# Visualize
img_with_points = image.copy()
draw = ImageDraw.Draw(img_with_points)

# Draw ALL points with different styles for positive/negative
radius = 8

# Draw sand points (red)
for point in sand_points:
    draw.ellipse([point[0]-radius, point[1]-radius, 
                  point[0]+radius, point[1]+radius], 
                 fill="red", outline="white", width=2)

# Draw air points (blue)
for point in air_points:
    draw.ellipse([point[0]-radius, point[1]-radius, 
                  point[0]+radius, point[1]+radius], 
                 fill="blue", outline="white", width=2)

# Draw background points (green)
for point in background_points:
    draw.ellipse([point[0]-radius, point[1]-radius, 
                  point[0]+radius, point[1]+radius], 
                 fill="green", outline="white", width=2)

# Create overlays
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].imshow(img_with_points)
axes[0, 0].set_title(f'Input Points: Red=Sand({len(sand_points)}), Blue=Air({len(air_points)}), Green=BG({len(background_points)})')
axes[0, 0].axis('off')

# Show individual masks
sand_overlay = np.array(image.copy())
sand_mask = sand_masks[0, 0].numpy() if len(sand_masks.shape) == 4 else sand_masks[0].numpy()
sand_overlay[sand_mask > 0.5] = [255, 0, 0]
axes[0, 1].imshow(sand_overlay)
axes[0, 1].set_title('Sand (excludes Air & BG)')
axes[0, 1].axis('off')

air_overlay = np.array(image.copy())
air_mask = air_masks[0, 0].numpy() if len(air_masks.shape) == 4 else air_masks[0].numpy()
air_overlay[air_mask > 0.5] = [0, 150, 255]
axes[1, 0].imshow(air_overlay)
axes[1, 0].set_title('Air (excludes Sand & BG)')
axes[1, 0].axis('off')

bg_overlay = np.array(image.copy())
bg_mask = background_masks[0, 0].numpy() if len(background_masks.shape) == 4 else background_masks[0].numpy()
bg_overlay[bg_mask > 0.5] = [0, 255, 0]
axes[1, 1].imshow(bg_overlay)
axes[1, 1].set_title('Background (excludes Sand & Air)')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Combined visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(img_with_points)
axes[0].set_title('Input Points')
axes[0].axis('off')

# Combined overlay
combined = np.array(image.copy())
combined[sand_mask > 0.5] = [255, 0, 0]
combined[air_mask > 0.5] = [0, 150, 255]
combined[bg_mask > 0.5] = [0, 255, 0]

axes[1].imshow(combined)
axes[1].set_title('Combined Segmentation with Multiple Points')
axes[1].axis('off')

plt.tight_layout()
plt.show()